# System Evaluation Notebook

**Purpose:** Smoke-test the modular pipeline in `src/` using lightweight settings to verify all functions work correctly.

**This notebook:**
- Tests data loading, embedding, and Milvus ingestion
- Verifies baseline RAG generation pipeline
- Tests enhanced pipeline (query rewriting + cross-encoder reranking)
- Validates evaluation formatting (SQuAD metrics + RAGAS dataset)
- Uses small samples (200 passages, 10 queries) for quick validation

**Note:** For the full 918-query SQuAD evaluation see `complete_analysis.ipynb`; for the scripted 100-query modular route use `python -m src.pipeline`.

## Run Plan Overview

The 100-query evaluation schedule is fixed:
1. Baseline prompt (`baseline`), top-1.
2. Verification prompt (`verification`), top-1.
3-5. MiniLM-L6 (384d) with top-3/5/10.
6-8. MiniLM-L3 (384d) with top-3/5/10.
9. Enhanced pipeline (rewrite+rerank, base top-10, rerank top-3).

RAGAS scoring follows for both naive and enhanced outputs (100-sample evaluation set).

In [1]:
from pathlib import Path
import sys
import os
import pandas as pd
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer

# Determine project root from notebook location
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Change to project root to access config files
os.chdir(PROJECT_ROOT)

from src.utils import load_text_table, load_queries
from src.naive_rag import (
    build_collection_schema,
    ensure_collection,
    encode_texts,
    build_rag_records,
    ingest_records,
    create_index,
    load_generation_resources,
    run_rag_generation,
)
from src.evaluation import format_for_squad_evaluation, build_ragas_dataset
from src.enhanced_rag import run_enhanced_rag_pipeline

In [2]:
# Configuration (use a tiny subset for smoke tests)
PASSAGES_PATH = "hf://datasets/rag-datasets/rag-mini-wikipedia/data/passages.parquet/part.0.parquet"
QUERIES_PATH = "hf://datasets/rag-datasets/rag-mini-wikipedia/data/test.parquet/part.0.parquet"

# Ensure data directory exists and use absolute path for Milvus
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
MILVUS_PATH = str(DATA_DIR / "rag_smoke_test.db")

COLLECTION_NAME = "rag_mini_smoke"
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L3-v2"
GENERATION_MODEL_NAME = "google/flan-t5-small"
SMOKE_TOP_K = 1
SMOKE_SAMPLE_SIZE = 10  # Use smaller sample for smoke test

print(f"Working directory: {os.getcwd()}")
print(f"Data directory: {DATA_DIR}")
print(f"Milvus database will be created at: {MILVUS_PATH}")

Working directory: /path/to/rag-eval
Data directory: /path/to/rag-eval/data
Milvus database will be created at: /path/to/rag-eval/data/rag_smoke_test.db


In [3]:
# Load sample data (smaller subset for smoke test)
passages = load_text_table(PASSAGES_PATH, "passage").head(200)
queries = load_queries(QUERIES_PATH).head(SMOKE_SAMPLE_SIZE)
print(f"Loaded {len(passages)} passages and {len(queries)} queries for smoke test.")

Loaded 200 passages and 10 queries for smoke test.


In [4]:
# Encode passages and ingest into temporary Milvus collection
print("Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Encoding passages...")
embeddings = encode_texts(embedding_model, passages["passage"], batch_size=64)
print(f"Embeddings shape: {embeddings.shape}")

# Create Milvus client
print(f"Creating Milvus client at: {MILVUS_PATH}")
client = MilvusClient(MILVUS_PATH)

# Build schema and create collection
print("Building collection schema...")
schema = build_collection_schema(embedding_dim=embeddings.shape[1])

print(f"Creating collection '{COLLECTION_NAME}'...")
ensure_collection(client, name=COLLECTION_NAME, schema=schema, drop_existing=True)

# Build and ingest records
print("Building records...")
records = build_rag_records(passages, embeddings)

print("Ingesting records...")
ingest_records(client, COLLECTION_NAME, records)

# Create index for efficient search
print("Creating index...")
create_index(client, COLLECTION_NAME)

print(f"✓ Ingested {len(records)} smoke-test passages with index.")

Loading embedding model...
Encoding passages...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (200, 384)
Creating Milvus client at: /path/to/rag-eval/data/rag_smoke_test.db


/path/to/rag-eval/.venv/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Building collection schema...
Creating collection 'rag_mini_smoke'...
Building records...
Ingesting records...
Creating index...
✓ Ingested 200 smoke-test passages with index.


In [5]:
# Baseline retrieval + generation
print("Loading generation resources...")
resources = load_generation_resources(GENERATION_MODEL_NAME)

print("Encoding questions...")
question_embeddings = encode_texts(embedding_model, queries["question"], batch_size=32, show_progress=False)

print("Running baseline RAG generation...")
baseline_outputs = run_rag_generation(
    queries,
    client=client,
    collection_name=COLLECTION_NAME,
    question_embeddings=question_embeddings,
    embedding_model=embedding_model,
    resources=resources,
    prompt_template="Answer using only the provided context.",
    top_k=SMOKE_TOP_K,
)
print(f"✓ Baseline generation complete. Generated {len(baseline_outputs['answer'])} answers.")

Loading generation resources...
Encoding questions...
Running baseline RAG generation...


Prompt='Answer using only the pr...' | top-1:   0%|          | 0/10 [00:00<?, ?it/s]

✓ Baseline generation complete. Generated 10 answers.


In [6]:
# Format for evaluation and build RAGAS dataset preview
print("Formatting for SQuAD evaluation...")
predictions, references = format_for_squad_evaluation(
    baseline_outputs["answer"], 
    baseline_outputs["ground_truths"]
)

print("Building sample RAGAS dataset...")
sample_dataset = build_ragas_dataset({
    "question": baseline_outputs["question"],
    "answer": baseline_outputs["answer"],
    "contexts": baseline_outputs["contexts"],
    "ground_truths": baseline_outputs["ground_truths"],
}, sample_size=min(5, len(queries)))

print(f"✓ SQuAD predictions: {len(predictions)}")
print(f"✓ RAGAS sample rows: {sample_dataset.num_rows}")

Formatting for SQuAD evaluation...
Building sample RAGAS dataset...
✓ SQuAD predictions: 10
✓ RAGAS sample rows: 5


In [7]:
# Enhanced pipeline smoke test (rewrite + rerank)
print("Running enhanced pipeline (rewrite + rerank)...")
enhanced_outputs = run_enhanced_rag_pipeline(
    queries,
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    resources=resources,
    base_top_k=5,
    rerank_top_k=3,
    prompt_template="Answer using only the provided context.",
    rewrite_strategy="recall",
)
print(f"✓ Enhanced pipeline complete. Generated {len(enhanced_outputs['answer'])} answers.")

Running enhanced pipeline (rewrite + rerank)...


Rewriting (recall):   0%|          | 0/10 [00:00<?, ?it/s]

Enhanced RAG:   0%|          | 0/10 [00:00<?, ?it/s]

✓ Enhanced pipeline complete. Generated 10 answers.


In [8]:
# Clean up temporary collection
print(f"Cleaning up temporary collection '{COLLECTION_NAME}'...")
try:
    client.drop_collection(collection_name=COLLECTION_NAME)
    print(f"✓ Dropped collection '{COLLECTION_NAME}'.")
except Exception as e:
    print(f"Warning: Could not drop collection: {e}")

print("\n" + "="*60)
print("SMOKE TEST COMPLETE")
print("="*60)
print(f"Baseline answers: {len(baseline_outputs['answer'])}")
print(f"Enhanced answers: {len(enhanced_outputs['answer'])}")
print("\nAll modular pipeline functions are working correctly!")

Cleaning up temporary collection 'rag_mini_smoke'...
✓ Dropped collection 'rag_mini_smoke'.

SMOKE TEST COMPLETE
Baseline answers: 10
Enhanced answers: 10

All modular pipeline functions are working correctly!
